In [8]:
import os
import json
import numpy as np
from tqdm import tqdm
import optuna

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# -------- Dataset --------
class RelativeSpeedDataset260D(Dataset):
    def __init__(self, annot_root, distance_json_path, max_items=None):
        self.items = []
        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"): continue
            sid = fname.replace(".json", "")
            if sid not in self.distances: continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)
            seq = ann.get("sequence", [])
            if len(seq) < 20: continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            tgt = np.array([f['TgtSpeed_ref'] for f in seq], dtype=np.float32)
            keys = [f"frame_{i+1:05d}" for i in range(len(seq))]
            dist = np.array([self.distances[sid].get(k, np.nan) for k in keys], dtype=np.float32)
            if len(dist) < 20: continue

            def smooth(x, w): return np.convolve(x, np.ones(w)/w, mode='same') if len(x) >= w else np.zeros_like(x)

            for i in range(len(seq) - 19):
                if max_items and len(self.items) >= max_items: return
                d, o, t = dist[i:i+20], own[i:i+20], tgt[i:i+20]
                if np.any(np.isnan(d)) or np.any(np.isnan(o)) or np.any(np.isnan(t)): continue

                rel_speed = t - o
                own_acc, d1, d2 = np.gradient(o), np.gradient(d), np.gradient(np.gradient(d))
                f3, f5, f7, f11 = smooth(d, 3), smooth(d, 5), smooth(d, 7), smooth(d, 11)
                f11_d1 = np.gradient(f11) if len(f11) >= 3 else np.zeros_like(f11)
                arrays = [d[:20], o[:20], own_acc[:20], d1[:20], d2[:20],
                          f3[:20], f5[:20], f7[:20], f11[:20], f11_d1[:20],
                          (f3 * d1)[:20], (f11 - f5)[:20], np.abs(d1[:20])]
                if any(a.shape[0] < 20 for a in arrays): continue
                feat = np.stack(arrays, axis=1)
                if feat.shape != (20, 13): continue
                target = np.mean(rel_speed)
                self.items.append((feat.astype(np.float32), target))

    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        feat, tgt = self.items[idx]
        return torch.tensor(feat), torch.tensor(tgt, dtype=torch.float32)

# -------- LSTM Model --------
class LSTM260D(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.fc(last).squeeze(1)

# -------- Objective --------
def objective(trial):
    hidden_size = trial.suggest_categorical("hidden_size", [64, 128, 256, 384])
    num_layers = trial.suggest_int("num_layers", 1, 4)
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])

    model = LSTM260D(input_size=13, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.SmoothL1Loss()

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    best_val_loss = float('inf')
    best_state_dict = None
    for epoch in range(30):
        model.train()
        for feats, tgts in train_loader:
            feats, tgts = feats.to(device), tgts.to(device)
            pred = model(feats)
            loss = criterion(pred, tgts)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for feats, tgts in val_loader:
                feats, tgts = feats.to(device), tgts.to(device)
                pred = model(feats)
                val_loss += criterion(pred, tgts).item() * feats.size(0)

        val_loss /= len(val_dataset)
        trial.report(val_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state_dict = model.state_dict()

    if best_state_dict:
        torch.save(best_state_dict, f"trial_{trial.number}_model.pth")

    return best_val_loss

# -------- 実行部 --------
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    full_dataset = RelativeSpeedDataset260D("./train_annotations", "../distance_ref_data.json", max_items=5000)
    train_idx, val_idx = train_test_split(list(range(len(full_dataset))), test_size=0.2, random_state=42)
    train_dataset = torch.utils.data.Subset(full_dataset, train_idx)
    val_dataset = torch.utils.data.Subset(full_dataset, val_idx)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=50)

    print("\n✅ 最適ハイパーパラメータ:")
    for k, v in study.best_params.items():
        print(f"{k}: {v}")
    print(f"best val loss: {study.best_value:.4f}")


[I 2025-05-16 01:59:05,475] A new study created in memory with name: no-name-b18813a0-8fd7-46db-9b48-c442d59e67ff
[I 2025-05-16 01:59:27,946] Trial 0 finished with value: 0.02331123460829258 and parameters: {'hidden_size': 384, 'num_layers': 2, 'dropout': 0.27727350942710616, 'lr': 3.576617696133752e-05, 'weight_decay': 0.0005623431836225396, 'batch_size': 32}. Best is trial 0 with value: 0.02331123460829258.
[I 2025-05-16 01:59:34,664] Trial 1 finished with value: 1.58147825050354 and parameters: {'hidden_size': 64, 'num_layers': 3, 'dropout': 0.42657528886875606, 'lr': 1.8780735893231764e-05, 'weight_decay': 1.0178437256978124e-05, 'batch_size': 128}. Best is trial 0 with value: 0.02331123460829258.
[I 2025-05-16 01:59:51,048] Trial 2 finished with value: 0.04403444866836071 and parameters: {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.21886542121471037, 'lr': 0.0007501606365518393, 'weight_decay': 0.00023059313936117887, 'batch_size': 32}. Best is trial 0 with value: 0.02331123


✅ 最適ハイパーパラメータ:
hidden_size: 384
num_layers: 2
dropout: 0.27727350942710616
lr: 3.576617696133752e-05
weight_decay: 0.0005623431836225396
batch_size: 32
best val loss: 0.0233


In [9]:
import os
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
from tqdm import tqdm

# -------- LSTM Model --------
class LSTM260D(nn.Module):
    def __init__(self, input_size=13, hidden_size=384, num_layers=2, dropout=0.277):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        last = out[:, -1, :]
        return self.fc(last).squeeze(1)

# -------- Inference Dataset --------
class InferenceDatasetLSTM260D(Dataset):
    def __init__(self, annot_root, distance_json_path):
        self.items = []
        self.seq_lens = {}

        with open(distance_json_path, encoding='utf-8') as f:
            self.distances = json.load(f)

        for fname in sorted(os.listdir(annot_root)):
            if not fname.endswith(".json"): continue
            sid = fname.replace(".json", "")
            if sid not in self.distances: continue

            with open(os.path.join(annot_root, fname), encoding='utf-8') as f:
                ann = json.load(f)
            seq = ann['sequence']
            self.seq_lens[sid] = len(seq)
            if len(seq) < 20: continue

            own = np.array([f['OwnSpeed'] for f in seq], dtype=np.float32)
            keys = sorted(self.distances[sid].keys())
            dist = np.array([self.distances[sid][k] for k in keys], dtype=np.float32)
            if len(dist) < 20: continue

            def smooth(x, w): return np.convolve(x, np.ones(w)/w, mode='same') if len(x) >= w else np.zeros_like(x)

            for i in range(len(seq) - 19):
                d = dist[i:i+20]
                o = own[i:i+20]
                if np.any(np.isnan(d)) or np.any(np.isnan(o)): continue

                own_acc, d1, d2 = np.gradient(o), np.gradient(d), np.gradient(np.gradient(d))
                f3, f5, f7, f11 = smooth(d, 3), smooth(d, 5), smooth(d, 7), smooth(d, 11)
                f11_d1 = np.gradient(f11) if len(f11) >= 3 else np.zeros_like(f11)
                arrays = [d[:20], o[:20], own_acc[:20], d1[:20], d2[:20],
                          f3[:20], f5[:20], f7[:20], f11[:20], f11_d1[:20],
                          (f3 * d1)[:20], (f11 - f5)[:20], np.abs(d1[:20])]
                if any(a.shape[0] < 20 for a in arrays): continue
                feat = np.stack(arrays, axis=1)
                if feat.shape != (20, 13): continue
                own_avg = np.mean(o)
                self.items.append((feat.astype(np.float32), own_avg, sid, i))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        feat, own_avg, sid, frame_idx = self.items[idx]
        return torch.tensor(feat), own_avg, sid, frame_idx

# -------- Predict Function --------
def predict_lstm_submission(model_path, annot_root, distance_json_path, save_path="submission.json",
                             hidden_size=384, num_layers=2, dropout=0.277):
    dataset = InferenceDatasetLSTM260D(annot_root, distance_json_path)
    loader = DataLoader(dataset, batch_size=64, shuffle=False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = LSTM260D(input_size=13, hidden_size=hidden_size, num_layers=num_layers, dropout=dropout).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    raw_preds = defaultdict(list)
    with torch.no_grad():
        for feats, own_speeds, sids, frame_idxs in tqdm(loader):
            feats = feats.to(device)
            own_speeds = own_speeds.numpy()
            preds = model(feats).cpu().numpy()
            abs_speeds = preds + own_speeds

            for sid, frame_idx, tgt in zip(sids, frame_idxs, abs_speeds):
                raw_preds[sid].append((frame_idx + 19, float(round(tgt, 3))))

    submission = {}
    for sid, pairs in raw_preds.items():
        pairs.sort()
        seq_len = dataset.seq_lens.get(sid, max(f for f, _ in pairs) + 1)
        pred_list = [0.0] * seq_len
        for idx, val in pairs:
            if idx < seq_len:
                pred_list[idx] = val
        for i in range(1, seq_len):
            if pred_list[i] == 0.0:
                pred_list[i] = pred_list[i-1]
        submission[sid] = pred_list

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)
    print(f"✅ submission.json 保存完了: {len(submission)} scenes")

# -------- 実行部 --------
if __name__ == "__main__":
    predict_lstm_submission(
        model_path="trial_0_model.pth",
        annot_root="../test2/test_annotations",
        distance_json_path="../testdistance/test_spline_smoothed_fixed.json",
        save_path="submission.json",
        hidden_size=384,
        num_layers=2,
        dropout=0.277
    )


100%|██████████| 395/395 [00:01<00:00, 251.34it/s]


✅ submission.json 保存完了: 239 scenes
